In [ ]:
!pip install --upgrade huggingface_hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import pandas as pd
from tqdm import tqdm

In [ ]:
df = pd.read_csv("sst_test_cases.csv")

In [ ]:
from vllm import LLM, SamplingParams

# Create an LLM.
llm = LLM(model='meta-llama/Llama-3.1-8B')

# Read fine tuned model from local
# llm = LLM(model='sft_model/')

In [ ]:
df_idx = 99
prompt = df.iloc[df_idx]['prompt']

instruction = f"{prompt}"

In [ ]:
sampling_params = SamplingParams(
    temperature=0.0, top_p=1.0, max_tokens=256
)

outputs = llm.generate([prompt], sampling_params)

# Print the outputs.
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}")
    print(f"Generated text: {generated_text!r}")


In [ ]:
from typing import Literal
def evaluate_vllm(
  vllm_model: LLM,
  prompts: list[str],
  eval_sampling_params: SamplingParams
) -> list[dict[str, str]] :
  """
  Evaluate a language model on a list of prompts,
  compute evaluation metrics, and serialize results to disk.
  """
  outputs = vllm_model.generate(prompts, eval_sampling_params)

  results = []

  for index, output in enumerate(outputs):
    results.append({"prompts_final": prompts[index], "output": output.outputs[0].text})

  return results

In [ ]:
prompts = []
final_results = []

for index, row in tqdm(df.iterrows()):
    prompt = row['prompt']

    instruction = f"{prompt}"
    prompts.append(prompt)

    if (index + 1) % 20 == 0:
        results = evaluate_vllm(llm, prompts, sampling_params)
        final_results += results
        prompts = []

In [3]:
import json

def write_jsonl(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        f.writelines(json.dumps(obj, ensure_ascii=False) + '\n' for obj in data)


write_jsonl(final_results, "sst.jsonl")